# ECC/EMC — Kaggle Training: EmoSet Semi-Supervised (SSL Only)

Semi-supervised FixMatch + EMC-adaptive threshold on EmoSet-118K.

## Before you run
1. **Enable GPU:** Settings → Session options → GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Public EmoSet-118K dataset in the sidebar.
3. **GitHub token secret:** `GITHUB_TOKEN` in Kaggle Settings → Secrets.
4. **Edit `LABEL_NUMS`** below (e.g., [4000, 8000] for EmoSet).

## What this runs
For each label count:
- **EMC-adaptive threshold** (`rule=ratio`): τ′ = τ·ε/EMC
- **FixMatch baseline** (`rule=fixed`): constant τ=0.95

## Results
Outputs go to `/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/`.
A zip cell at the end creates `emoset_ssl_results.zip` for one-click download.


In [ ]:
# ===================== CONFIGURE THESE =====================
# Label counts for EmoSet SSL (paper uses 4000, 8000)
LABEL_NUMS = [4000, 8000]   # <-- EDIT as needed

# Public EmoSet-118K dataset
EMOSET_DATASET_SLUG = "georgeliil/emoset"   # <-- REPLACE if different

# ==========================================================

print("Label nums:", LABEL_NUMS)
print("EmoSet dataset slug:", EMOSET_DATASET_SLUG)

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone repo using GitHub token secret (added in Kaggle Settings → Secrets)
import os
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{secret_token}@github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"
except Exception:
    repo_url = "https://github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"

dest = "/kaggle/working/ecc-ver"

if not os.path.exists(dest):
    subprocess.run(["git", "clone", repo_url, dest], check=True)
    print("Cloned repo to", dest)
else:
    print("Repository already exists at", dest)

os.chdir(dest)
print("Current dir:", os.getcwd())

In [ ]:
# Install ecc-ver package
!pip -q install -e .

In [ ]:
# Symlink EmoSet dataset into data/ layout
import os
from pathlib import Path

os.makedirs("data", exist_ok=True)

# Robustly find EmoSet/ anywhere under /kaggle/input
found_images = False
found_splits = False
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 5:
        dirs[:] = []
        continue
    for d in dirs:
        if d in ("image", "EmoSet") and not found_images:
            src = os.path.join(root, d)
            if any((Path(src) / c).is_dir() for c in ["amusement", "anger", "awe", "contentment", "disgust", "excitement", "fear", "sadness"]):
                dst = "data/EmoSet"
                if not os.path.exists(dst):
                    os.symlink(src, dst)
                    print(f"Linked EmoSet images -> data/EmoSet (from {src})")
                found_images = True
        if d == "EmoSet_splits" and not found_splits:
            src = os.path.join(root, d)
            dst = "data/EmoSet_splits"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked EmoSet splits -> data/EmoSet_splits (from {src})")
            found_splits = True

if not found_images:
    print("WARNING: Could not find EmoSet image directory under /kaggle/input")
if not found_splits:
    print("WARNING: Could not find EmoSet_splits under /kaggle/input")

print("\nData layout:")
!find data -maxdepth 3 | head -50

In [ ]:
# Quick smoke test
!python -m pytest -q

In [ ]:
# ===================== SEMI-SUPERVISED (EmoSet, Section 4.2.1) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for n_labeled in LABEL_NUMS:
    for rule in ["ratio", "fixed"]:
        print(f"\n=== SSL n={n_labeled}, rule={rule} (EMC-adaptive vs FixMatch) ===")
        cmd = f"python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled={n_labeled} ssl.rule={rule} --set data.root=data/EmoSet data.splits_dir=data/EmoSet_splits output_dir=/kaggle/working/runs/emoset_ssl_n{n_labeled}_{rule}"
        print("Running:", cmd)
        !{cmd}

print("SSL done.")

In [ ]:
# ===================== PREPARE RESULTS ZIP FOR ONE-CLICK DOWNLOAD =====================
import zipfile, os, glob, shutil

runs_dir = "/kaggle/working/runs"
zip_path = "/kaggle/working/emoset_ssl_results.zip"

run_dirs = []
for n_labeled in LABEL_NUMS:
    for rule in ["ratio", "fixed"]:
        run_dirs.append(os.path.join(runs_dir, f"emoset_ssl_n{n_labeled}_{rule}"))

run_dirs = [d for d in run_dirs if os.path.exists(d)]

if not run_dirs:
    print("No emoset_ssl_* run directories found under", runs_dir)
else:
    stage_root = os.path.join(runs_dir, "_zip_stage")
    os.makedirs(stage_root, exist_ok=True)
    
    for run_dir in run_dirs:
        run_name = os.path.basename(run_dir)
        stage = os.path.join(stage_root, run_name)
        os.makedirs(stage, exist_ok=True)
        
        best_pt = os.path.join(run_dir, "best.pt")
        results_json = os.path.join(run_dir, "results.json")
        history_jsonl = os.path.join(run_dir, "history.jsonl")
        config_json = os.path.join(run_dir, "config.json")
        
        if os.path.exists(best_pt):
            shutil.copy2(best_pt, os.path.join(stage, "best.pt"))
        if os.path.exists(results_json):
            shutil.copy2(results_json, os.path.join(stage, "results.json"))
        if os.path.exists(history_jsonl):
            shutil.copy2(history_jsonl, os.path.join(stage, "history.jsonl"))
        if os.path.exists(config_json):
            shutil.copy2(config_json, os.path.join(stage, "config.json"))
        
        print(f"Staged {run_name}")
    
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for stage in sorted(glob.glob(os.path.join(stage_root, "emoset_ssl_*"))):
            run_name = os.path.basename(stage)
            for fname in sorted(os.listdir(stage)):
                fpath = os.path.join(stage, fname)
                zf.write(fpath, arcname=os.path.join(run_name, fname))
    
    shutil.rmtree(stage_root, ignore_errors=True)
    
    print(f"\nCreated {zip_path}")
    print("Contents:")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            print(f"  {info.filename} ({info.file_size:,} bytes)")
    
    print("\nDownload from Output panel (right sidebar → Output).")

## Resuming after session timeout

Kaggle free sessions are capped at ~12 hours. Every epoch saves `checkpoint.pt` to:
1. The run's `output_dir` (`/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/`), and
2. `PERSISTENT_DIR` (your mounted Kaggle Dataset, if set — see notebook 1).

If a session dies, re-run the same cell — it auto-detects `checkpoint.pt` in the output dir
and resumes from the last epoch.

To force a specific checkpoint, add `--set resume_from=/kaggle/working/runs/emoset_ssl_n<labels>_<rule>/checkpoint.pt`.

## Download results

Run the zip cell above to create `emoset_ssl_results.zip` containing, for each run:
- `best.pt` — best-on-val model weights
- `checkpoint.pt` — resumable training checkpoint
- `results.json` — test metrics (ACC, ECC, EMC, ACC2)
- `history.jsonl` — per-epoch training log
- `config.json` — exact config used

Download the zip from the notebook output panel before the session ends.
